Cleaning 02_nav_history.csv

In [1]:
import pandas as pd
# Load CSV
df = pd.read_csv("../data/raw/02_nav_history.csv")


In [2]:
# Parse date column to datetime
df['date'] = pd.to_datetime(df['date'], errors='coerce')


In [3]:
# Remove rows with invalid dates
df = df.dropna(subset=['date'])


In [4]:
# Sort by amfi_code and date
df = df.sort_values(by=['amfi_code', 'date'])


In [5]:
# Remove duplicate records
df = df.drop_duplicates()

In [6]:
# Forward-fill missing NAV values within each fund
df['nav'] = df.groupby('amfi_code')['nav'].ffill()

In [7]:
# Validate NAV > 0
df = df[df['nav'] > 0]


In [8]:
# Reset index
df = df.reset_index(drop=True)


In [9]:
# Save cleaned file
df.to_csv("nav_history_cleaned.csv", index=False)

In [10]:
print("Data cleaning completed successfully!")
print(f"Total records after cleaning: {len(df)}")

Data cleaning completed successfully!
Total records after cleaning: 46000


Cleaning 08_investor_transactions.csv

In [11]:
import pandas as pd
import re

# Load CSV
df = pd.read_csv("../data/raw/08_investor_transactions.csv")


In [12]:
# 1. Fix Date Formats
# -----------------------------
df['transaction_date'] = pd.to_datetime(
    df['transaction_date'],
    errors='coerce',
    dayfirst=True
)

In [13]:
# Remove invalid dates
df = df.dropna(subset=['transaction_date'])


In [14]:
# 2. Standardize Transaction Type
# -----------------------------
def standardize_transaction_type(txn):
    if pd.isna(txn):
        return None

    txn = str(txn).strip().lower()

    if re.search(r'sip|systematic', txn):
        return 'SIP'
    elif re.search(r'lump|purchase|investment', txn):
        return 'Lumpsum'
    elif re.search(r'redemption|redeem|withdraw', txn):
        return 'Redemption'
    else:
        return 'Unknown'

df['transaction_type'] = df['transaction_type'].apply(
    standardize_transaction_type
)



In [19]:
# 3. Validate Amount > 0
# -----------------------------
df['amount_inr'] = pd.to_numeric(df['amount_inr'], errors='coerce')

df = df[df['amount_inr'] > 0]

In [20]:
# 4. Check and Standardize KYC Status
# -----------------------------
valid_kyc = {
    'yes': 'Verified',
    'verified': 'Verified',
    'complete': 'Verified',
    'completed': 'Verified',
    'no': 'Pending',
    'pending': 'Pending',
    'in progress': 'Pending',
    'rejected': 'Rejected',
    'failed': 'Rejected'
}

df['kyc_status'] = (
    df['kyc_status']
    .astype(str)
    .str.strip()
    .str.lower()
    .map(valid_kyc)
)


In [21]:
# Keep only valid KYC statuses
df = df[df['kyc_status'].isin(['Verified', 'Pending', 'Rejected'])]


In [22]:
# 5. Remove Duplicates
# -----------------------------
df = df.drop_duplicates()


In [23]:
# 6. Save Cleaned File
# -----------------------------
df.to_csv("../data/Processed/investor_transactions_cleaned.csv", index=False)

print("Cleaning completed successfully!")
print("Total records:", len(df))

Cleaning completed successfully!
Total records: 32778


Cleaning 07_scheme_performance.csv

In [24]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("../data/raw/07_scheme_performance.csv")


In [27]:
# 1. Validate Return Values
# -----------------------------------
return_cols = ['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct']

for col in return_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')


In [26]:
import pandas as pd

# Load the CSV file
df = pd.read_csv('../data/raw/07_scheme_performance.csv')

# Print column names as a list
print(df.columns.tolist())


['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade']


In [28]:
# Remove rows where all return values are invalid
df = df.dropna(subset=return_cols, how='all')

In [29]:
# 2. Validate Sharpe Ratio
# -----------------------------------
df['sharpe_ratio'] = pd.to_numeric(
    df['sharpe_ratio'],
    errors='coerce'
)


In [30]:
# Flag negative Sharpe ratios
df['Negative_Sharpe_Flag'] = np.where(
    df['sharpe_ratio'] < 0,
    'Yes',
    'No'
)


In [31]:
# 3. Validate Expense Ratio
# -----------------------------------
df['expense_ratio_pct'] = pd.to_numeric(
    df['expense_ratio_pct'],
    errors='coerce'
)


In [32]:
# Flag ratios outside industry range
df['Expense_Ratio_Valid'] = np.where(
    (df['expense_ratio_pct'] >= 0.1) &
    (df['expense_ratio_pct'] <= 2.5),
    'Valid',
    'Out_of_Range'
)

In [33]:
# 4. Remove Duplicate Records
# -----------------------------------
df = df.drop_duplicates()

In [34]:
# 5. Save Cleaned Dataset
# -----------------------------------
df.to_csv("clean_performance.csv", index=False)


In [35]:
# Summary
print("Cleaning completed successfully!")
print(f"Total records: {len(df)}")
print(f"Negative Sharpe Ratios: {(df['Negative_Sharpe_Flag']=='Yes').sum()}")
print(f"Out-of-Range Expense Ratios: {(df['Expense_Ratio_Valid']=='Out_of_Range').sum()}")

Cleaning completed successfully!
Total records: 40
Negative Sharpe Ratios: 0
Out-of-Range Expense Ratios: 0
